# Data Preprocessing – Data Quality

### Dataset: `data_center_hybrid.csv`

**Tasks:**
1. Data Smoothing using Binning
2. Data Normalization
3. Redundancy Analysis using Pearson Correlation
4. Redundancy Analysis using Chi-Square Test
5. Discretization using Intuitive Partitioning (3-4-5 Rule)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, chi2_contingency
import math

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DAV/data_center_hybrid.csv')

print('Dataset shape:', df.shape)

Dataset shape: (126770, 14)


## Block 1: Implement Data Smoothing

Data smoothing reduces small fluctuations/noise in numerical data. Here, the first 20 `PUE` values are sorted and divided into 4 equal-frequency bins. Each value is replaced by the mean of its bin.

In [4]:
# Select first 20 PUE values
pue = df['PUE'].head(20).sort_values().reset_index(drop=True)

# Divide into 4 bins and replace values by bin mean
bins = np.array_split(pue, 4)
smoothed = pd.concat([pd.Series([b.mean()] * len(b)) for b in bins], ignore_index=True)

print('Original sorted values:')
print(pue.tolist())

print('\nSmoothed values:')
print(smoothed.tolist())

Original sorted values:
[1.668, 1.696, 1.721, 1.728, 1.759, 1.792, 1.796, 1.815, 1.842, 1.869, 1.87, 1.88, 1.892, 1.897, 1.902, 1.928, 1.931, 1.961, 1.967, 1.975]

Smoothed values:
[1.7144, 1.7144, 1.7144, 1.7144, 1.7144, 1.8228000000000002, 1.8228000000000002, 1.8228000000000002, 1.8228000000000002, 1.8228000000000002, 1.8881999999999999, 1.8881999999999999, 1.8881999999999999, 1.8881999999999999, 1.8881999999999999, 1.9524000000000001, 1.9524000000000001, 1.9524000000000001, 1.9524000000000001, 1.9524000000000001]


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


## Block 2: Implement Data Normalization

**Min-Max normalization:** converts values to the range 0 to 1.

**Z-Score normalization:** transforms values so that the mean is approximately 0 and the standard deviation is 1.

In [5]:
pue = df['PUE']

# Min-Max normalization
df['PUE_MinMax'] = (pue - pue.min()) / (pue.max() - pue.min())

# Z-Score normalization
df['PUE_ZScore'] = (pue - pue.mean()) / pue.std()

display(df[['PUE', 'PUE_MinMax', 'PUE_ZScore']].head(10))

,PUE,PUE_MinMax,PUE_ZScore
0,1.975,0.973489,1.767275
1,1.967,0.965005,1.725353
2,1.928,0.923648,1.520986
3,1.897,0.890774,1.358540
4,1.869,0.861082,1.211814
5,1.842,0.832450,1.070329
6,1.815,0.803818,0.928844
7,1.796,0.783669,0.829280
8,1.792,0.779427,0.808320
9,1.759,0.744433,0.635393


## Block 3: Redundancy Analysis using Pearson Correlation

Pearson correlation measures the linear relationship between two numerical attributes.

- Correlation close to **+1 or -1** → strong relationship
- Correlation close to **0** → weak relationship
- A very high absolute correlation can indicate possible redundancy.

In [6]:
x = df['Estimated_Capacity_MW']
y = df['Daily_Electricity_Usage_MWh']

correlation, p_value = pearsonr(x, y)

print('Pearson correlation:', round(correlation, 4))
print('P-value:', p_value)

if abs(correlation) > 0.9:
    print('The attributes are highly correlated and may be redundant.')
else:
    print('The attributes are not highly correlated.')

Pearson correlation: 0.9835
P-value: 0.0
The attributes are highly correlated and may be redundant.


## Block 4: Redundancy Analysis using Chi-Square Test

The Chi-Square test checks whether two categorical attributes are independent.

- **p-value < 0.05** → attributes are related (dependent)
- **p-value >= 0.05** → no significant relationship is found (independent)

In [7]:
# Create contingency table
table = pd.crosstab(df['Facility_Type'], df['Cooling_System_Type'])

chi2, p_value, dof, expected = chi2_contingency(table)

print('Contingency Table:')
display(table)

print('Chi-Square value:', round(chi2, 4))
print('P-value:', p_value)
print('Degrees of freedom:', dof)

if p_value < 0.05:
    print('The attributes are dependent (related).')
else:
    print('The attributes are independent.')

Contingency Table:


Cooling_System_Type,Air Cooled,Evaporative,Liquid Cooled
Facility_Type,,,
Colocation,4823,6979,714
Enterprise/Standard,72877,32312,0
Hyperscale/AI,1841,6321,903


Chi-Square value: 18155.9652
P-value: 0.0
Degrees of freedom: 4
The attributes are dependent (related).


## Block 5: Discretization using Intuitive Partitioning

Discretization converts a continuous numerical attribute into intervals (bins). Here, `Estimated_Capacity_MW` is divided using the textbook **3-4-5 rule** to create simple, readable ranges.

In [8]:
low = df['Estimated_Capacity_MW'].min()
high = df['Estimated_Capacity_MW'].max()
data_range = high - low

# Find the most significant digit of the range
place = 10 ** math.floor(math.log10(data_range))
leading_digit = int(data_range / place)

# 3-4-5 rule
if leading_digit in [1, 5]:
    parts = 5
elif leading_digit in [2, 4, 8]:
    parts = 4
else:
    parts = 3

# Create bin edges
start = math.floor(low / place) * place
end = math.ceil(high / place) * place
edges = np.linspace(start, end, parts + 1)
edges = np.round(edges, 2)

labels = [f'{edges[i]} to {edges[i+1]}' for i in range(len(edges) - 1)]

df['Capacity_Bucket'] = pd.cut(
    df['Estimated_Capacity_MW'],
    bins=edges,
    labels=labels,
    include_lowest=True
)

print('Minimum capacity:', low)
print('Maximum capacity:', high)
print('Number of intervals:', parts)
print('Bin edges:', edges.tolist())

print('\nNumber of facilities in each bucket:')
print(df['Capacity_Bucket'].value_counts().sort_index())

Minimum capacity: 1.0
Maximum capacity: 562.89
Number of intervals: 5
Bin edges: [0.0, 120.0, 240.0, 360.0, 480.0, 600.0]

Number of facilities in each bucket:
Capacity_Bucket
0.0 to 120.0      121430
120.0 to 240.0      3988
240.0 to 360.0      1059
360.0 to 480.0       260
480.0 to 600.0        33
Name: count, dtype: int64


## Block 6: Save Preprocessed Dataset

The normalized and discretized columns are saved into a new CSV file.

In [9]:
output_file = 'data_center_hybrid_preprocessed.csv'
df.to_csv(output_file, index=False)
print('Preprocessed dataset saved as:', output_file)

Preprocessed dataset saved as: data_center_hybrid_preprocessed.csv
